# Post Training Quantization : KSW 
## Before Quantization
1) Dataset, Train Dataloader, Test Dataloader, Transforms
    - Dataset is MNIST
    - Data loader lakes care of batching etc
    - Also some transforms to apply on the dataset
2) Define Model: 
     - User defined. 3 linear layers + RELU
3) Define: Training Loop
  Choose the following
   - Choose Loss : Cross Entropy Loss
   - Choose Optimizer: Adam Optimizer
   - Choose Number of Iterations
   - set model to train mode: net.train(). this enables dropout and uses batch norm 
4) Define: Test Loop
5) Define: Getting Model Size
6) Training: run the training loop
7) Testing: run the testing loop
8) Get the Size of Trained model
   
   

## Import Necessary Libraries

In [1]:
# !pip install datasets
# !pip install --upgrade datasets

In [2]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import os
import numpy as np

from tqdm import tqdm
from pathlib import Path
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

## Load MNIST dataset

In [3]:
# Make torch deterministic
_ = torch.manual_seed(0)

In [4]:
# ----------------------------
# 1️⃣ Transform
# ----------------------------
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# ----------------------------
# 2️⃣ PyTorch MNIST wrapper for HF datasets v4.6.0
# ----------------------------
class HFMNISTDataset(Dataset):
    def __init__(self, split="train"):
        self.dataset = load_dataset("mnist", split=split)
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        example = self.dataset[idx]
        img = np.array(example["image"], dtype=np.uint8)
        # flatten extra dims if necessary
        if img.ndim > 2:
            img = img.squeeze()
        # ensure correct shape
        if img.shape != (28,28):
            img = img.reshape(28,28)
        # PIL grayscale image
        img = Image.fromarray(img, mode='L')
        img = self.transform(img)
        label = torch.tensor(example["label"], dtype=torch.long)
        return img, label

# ----------------------------
# 3️⃣ Create DataLoaders
# ----------------------------
train_dataset = HFMNISTDataset("train")
test_dataset  = HFMNISTDataset("test")

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

# ----------------------------
# 4️⃣ Test batch
# ----------------------------
x, y = next(iter(train_loader))
print(x.shape, y.shape)  # torch.Size([32,1,28,28]), torch.Size([32])

torch.Size([32, 1, 28, 28]) torch.Size([32])


In [5]:
# define device
device = "cpu"

## Define: Model

In [6]:
class VerySimpleNet(nn.Module):
    def __init__(self, hidden_size_1=100, hidden_size_2=100):
        super(VerySimpleNet,self).__init__()
        self.linear1 = nn.Linear(28*28, hidden_size_1) 
        self.linear2 = nn.Linear(hidden_size_1, hidden_size_2) 
        self.linear3 = nn.Linear(hidden_size_2, 10)
        self.relu = nn.ReLU()

    def forward(self, img):
        x = img.view(-1, 28*28)
        x = self.relu(self.linear1(x))
        x = self.relu(self.linear2(x))
        x = self.linear3(x)
        return x

In [7]:
vsnet = VerySimpleNet().to(device)

## Define: Training Loop

In [8]:
def train(train_loader, model, epochs=5, total_iterations_all_epochs_limit=None):
    cross_el = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    total_iterations_all_epochs = 0

    for epoch in range(epochs):
        # puts the model in training mode
        # doesnt do the actual training itself, It enables the right behaviour in training mode. 
        # i.e. enables dropout, 
        # enables batch normalization to use the current batch's statistics : mean and variance
        model.train()

        total_loss_this_epoch = 0
        # num_iterations_this_epoch is same as the number_of_batches_this_epoch
        num_iterations_this_epoch = 0

        # data loader is the train_loader wrapped in tqdm, so it shows a progress bar
        # But it is really the same
        data_loader = tqdm(train_loader, desc=f'Epoch {epoch+1}')
        if total_iterations_all_epochs_limit is not None:
            data_loader.total = total_iterations_all_epochs_limit

        # Iterate through every batch (within this epoch)
        for batch in data_loader:

            num_iterations_this_epoch += 1
            total_iterations_all_epochs += 1

            # Get the (x,y) from the batch and deploy them to the correct device
            x, y = batch
            x = x.to(device)
            y = y.to(device)

            # Step 0. Zero out the previous gradients
            optimizer.zero_grad()
            if total_iterations_all_epochs <3:
                print("\n\n ------- print parameters after optimizer.zero_grad()-------")
                print_parameters(vsnet)
                            
            # Step 1. Forward Pass
            # model(x) performs a forward pass over the entire batch at once, not one sample at a time.
            # PyTorch operations are vectorized, so computations happen in parallel on the batch (especially on GPU).

            output = model(x.view(-1, 28*28))

            # Step 2. Calculate the Loss
            loss = cross_el(output, y)
            total_loss_this_epoch += loss.item()
            # This is the running average loss (of all the batches seen) in this epoch
            running_avg_loss = total_loss_this_epoch / num_iterations_this_epoch
            # data_loader is the tqdm wrapper around train_loader
            # This updates the tqdm loss bar with the running average loss so far this epoch
            data_loader.set_postfix(loss=running_avg_loss)

            # Step 3. Backpropagate the loss. Calcualte the gradients
            loss.backward()

            # Step 4. Update the weights based on the gradients
            optimizer.step()
            if total_iterations_all_epochs <3:
                print("\n\n ------- print parameters after optimizer.step()-------")
                print_parameters(vsnet)

            # Early stopping based on iteration count
            if total_iterations_all_epochs_limit is not None and total_iterations_all_epochs >= total_iterations_all_epochs_limit:
                return
                

## Define: Test Loop

In [9]:
def test(model: nn.Module, total_iterations_all_epochs_all_epochs: int = None):
    correct = 0
    total = 0

    iterations = 0

    # This puts the model in eval mode
    # i) disables dropout, ii) batch normalization will use the stored mean and variance (not the current batch statistics)
    model.eval()

    with torch.no_grad():
        for batch in tqdm(test_loader, desc='Testing'):
            x, y = batch
            x = x.to(device)
            y = y.to(device)
            output = model(x.view(-1, 784))
            for idx, i in enumerate(output):
                if torch.argmax(i) == y[idx]:
                    correct +=1
                total +=1
            iterations += 1
            if total_iterations_all_epochs_all_epochs is not None and iterations >= total_iterations_all_epochs_all_epochs:
                break
    print(f'Accuracy: {round(correct/total, 3)}')

## Define: Other Helper Functions

In [10]:
def print_size_of_model(model):
    torch.save(model.state_dict(), "temporary_delete_me.p")
    print('Size (KB):', os.path.getsize("temporary_delete_me.p")/1e3)
    os.remove('temporary_delete_me.p')

def print_parameters(model):
    print("param.requires_grad: True (by default)", )
    print("param.grad         : None (initially when not training ???)")
    print("param.grad_fn      : None (parameters are leaf tensors , and will be None always")
    
    for name, param in model.named_parameters():
        print("\n ---------name:", name, "----------")
        print("param.data  : not printing data values, too big")    
        print("param.dtype :", param.dtype)
        print("param.shape :", param.shape)
        print("param.device:", param.device)    
        print("param.requires_grad:", param.requires_grad)
        print("param.grad         :", param.grad)
        print("param.grad_fn      :", param.grad_fn)

## Before Quantization : Print the models parameters and requires_grad

In [11]:
print_parameters(vsnet)

param.requires_grad: True (by default)
param.grad         : None (initially when not training ???)
param.grad_fn      : None (parameters are leaf tensors , and will be None always

 ---------name: linear1.weight ----------
param.data  : not printing data values, too big
param.dtype : torch.float32
param.shape : torch.Size([100, 784])
param.device: cpu
param.requires_grad: True
param.grad         : None
param.grad_fn      : None

 ---------name: linear1.bias ----------
param.data  : not printing data values, too big
param.dtype : torch.float32
param.shape : torch.Size([100])
param.device: cpu
param.requires_grad: True
param.grad         : None
param.grad_fn      : None

 ---------name: linear2.weight ----------
param.data  : not printing data values, too big
param.dtype : torch.float32
param.shape : torch.Size([100, 100])
param.device: cpu
param.requires_grad: True
param.grad         : None
param.grad_fn      : None

 ---------name: linear2.bias ----------
param.data  : not printing dat

## Before Quantization : Train the Model

In [12]:
MODEL_FILENAME = 'verysimplenet_ptq.pt'

if Path(MODEL_FILENAME).exists():
    vsnet.load_state_dict(torch.load(MODEL_FILENAME))
    print('Loaded model from disk')
else:
    train(train_loader, vsnet, epochs=1)
    # Save the model to disk
    torch.save(vsnet.state_dict(), MODEL_FILENAME)

Loaded model from disk


## Before Quantization: Weights & Size of the Model 

In [13]:
print('-----------------------------------------')
print('Size of the model before quantization')
print('-----------------------------------------')
print_size_of_model(vsnet)



# Print the weights matrix of the model before quantization
print('\n\n')
print('-----------------------------------------')
print('Weights before quantization')
print('-----------------------------------------')
print('-------linear1--------')
print(vsnet.linear1.weight)
print(vsnet.linear1.weight.dtype)
print('\n-------linear2--------')
print(vsnet.linear2.weight)
print(vsnet.linear2.weight.dtype)

-----------------------------------------
Size of the model before quantization
-----------------------------------------
Size (KB): 361.088



-----------------------------------------
Weights before quantization
-----------------------------------------
-------linear1--------
Parameter containing:
tensor([[-0.0176,  0.0058, -0.0045,  ..., -0.0188, -0.0143, -0.0386],
        [-0.0078, -0.0034, -0.0097,  ...,  0.0052, -0.0430, -0.0300],
        [ 0.0329, -0.0121, -0.0087,  ..., -0.0268,  0.0079, -0.0031],
        ...,
        [-0.0275, -0.0310, -0.0264,  ..., -0.0182,  0.0138,  0.0005],
        [ 0.0029, -0.0112,  0.0098,  ...,  0.0213,  0.0072,  0.0223],
        [ 0.0120,  0.0347, -0.0144,  ...,  0.0303, -0.0096,  0.0003]],
       requires_grad=True)
torch.float32

-------linear2--------
Parameter containing:
tensor([[ 0.0534,  0.0360,  0.0552,  ..., -0.1173,  0.1025,  0.0142],
        [-0.0187, -0.0194,  0.1173,  ..., -0.0387,  0.0577,  0.0638],
        [-0.2215,  0.1117, -0.0344,  .

## Before Quantization: Test Model & Accuracy

In [14]:
print('Accuracy of the model before quantization: ')
test(vsnet)

Accuracy of the model before quantization: 


Testing: 100%|███████████████████████████████| 313/313 [00:02<00:00, 126.94it/s]

Accuracy: 0.958


# Insert min-max observers in the model
- QuantStub : converts floating-point tensors to quantized tensors (e.g., int8) before the network computation. Below it converts the input tensor x from floating point to int8
- DeQuantStub : Converts output back to float for loss calculation or evaluation.
- only the input activations (the tensor x) are quantized.
- The linear layers (linear1, linear2, linear3) themselves are still in floating point. Their weights and biases are not quantized yet.
- Hence the model below is "Activation-Quantized" only i.e only the input activation tensor x is quantized


In [15]:
class QuantizedVerySimpleNet(nn.Module):
    def __init__(self, hidden_size_1=100, hidden_size_2=100):
        super(QuantizedVerySimpleNet,self).__init__()
        self.quantize = torch.quantization.QuantStub()
        self.linear1 = nn.Linear(28*28, hidden_size_1) 
        self.linear2 = nn.Linear(hidden_size_1, hidden_size_2) 
        self.linear3 = nn.Linear(hidden_size_2, 10)
        self.relu = nn.ReLU()
        self.dequantize = torch.quantization.DeQuantStub()

    def forward(self, img):
        x = img.view(-1, 28*28)
        x = self.quantize(x)
        x = self.relu(self.linear1(x))
        x = self.relu(self.linear2(x))
        x = self.linear3(x)
        x = self.dequantize(x)
        return x

## Full Static Quantization
- i)   specify a config: tells how to quantize weights and the activations
- ii)  fuse layers where possible like : linear and relu 
- iii) run calibration with representative data
- iv)  quantize the model: convert the original weights to linear weights

In [16]:
vsnet_quantized = QuantizedVerySimpleNet().to(device)
# Copy weights from unquantized model
vsnet_quantized.load_state_dict(vsnet.state_dict())
vsnet_quantized.eval()

vsnet_quantized.qconfig = torch.ao.quantization.default_qconfig
vsnet_quantized = torch.ao.quantization.prepare(vsnet_quantized) # Insert observers
vsnet_quantized

QuantizedVerySimpleNet(
  (quantize): QuantStub(
    (activation_post_process): MinMaxObserver(min_val=inf, max_val=-inf)
  )
  (linear1): Linear(
    in_features=784, out_features=100, bias=True
    (activation_post_process): MinMaxObserver(min_val=inf, max_val=-inf)
  )
  (linear2): Linear(
    in_features=100, out_features=100, bias=True
    (activation_post_process): MinMaxObserver(min_val=inf, max_val=-inf)
  )
  (linear3): Linear(
    in_features=100, out_features=10, bias=True
    (activation_post_process): MinMaxObserver(min_val=inf, max_val=-inf)
  )
  (relu): ReLU()
  (dequantize): DeQuantStub()
)

# Calibrate the model using the test set

In [17]:
test(vsnet_quantized)

Testing: 100%|████████████████████████████████| 313/313 [00:03<00:00, 98.03it/s]

Accuracy: 0.958


In [18]:
print(f'Check statistics of the various layers')
vsnet_quantized

Check statistics of the various layers


QuantizedVerySimpleNet(
  (quantize): QuantStub(
    (activation_post_process): MinMaxObserver(min_val=-0.4242129623889923, max_val=2.821486711502075)
  )
  (linear1): Linear(
    in_features=784, out_features=100, bias=True
    (activation_post_process): MinMaxObserver(min_val=-30.252206802368164, max_val=22.892858505249023)
  )
  (linear2): Linear(
    in_features=100, out_features=100, bias=True
    (activation_post_process): MinMaxObserver(min_val=-16.78236198425293, max_val=16.252704620361328)
  )
  (linear3): Linear(
    in_features=100, out_features=10, bias=True
    (activation_post_process): MinMaxObserver(min_val=-24.6287784576416, max_val=18.362592697143555)
  )
  (relu): ReLU()
  (dequantize): DeQuantStub()
)

# Quantize the model using the statistics collected

In [19]:
vsnet_quantized = torch.ao.quantization.convert(vsnet_quantized)

In [20]:
print(f'Check statistics of the various layers')
vsnet_quantized

Check statistics of the various layers


QuantizedVerySimpleNet(
  (quantize): Quantize(scale=tensor([0.0256]), zero_point=tensor([17]), dtype=torch.quint8)
  (linear1): QuantizedLinear(in_features=784, out_features=100, scale=0.4184650778770447, zero_point=72, qscheme=torch.per_tensor_affine)
  (linear2): QuantizedLinear(in_features=100, out_features=100, scale=0.26011863350868225, zero_point=65, qscheme=torch.per_tensor_affine)
  (linear3): QuantizedLinear(in_features=100, out_features=10, scale=0.3385147452354431, zero_point=73, qscheme=torch.per_tensor_affine)
  (relu): ReLU()
  (dequantize): DeQuantize()
)

# Print weights of the model after quantization

In [21]:
# Print the weights matrix of the model after quantization
print('Weights after quantization')
print(torch.int_repr(vsnet_quantized.linear1.weight()))

Weights after quantization
tensor([[ -6,   2,  -2,  ...,  -7,  -5, -13],
        [ -3,  -1,  -3,  ...,   2, -15, -10],
        [ 11,  -4,  -3,  ...,  -9,   3,  -1],
        ...,
        [-10, -11,  -9,  ...,  -6,   5,   0],
        [  1,  -4,   3,  ...,   7,   2,   8],
        [  4,  12,  -5,  ...,  10,  -3,   0]], dtype=torch.int8)


# Compare the dequantized weights and the original weights

In [22]:
print('Original weights: ')
print(vsnet.linear1.weight)
print('')
print(f'Dequantized weights: ')
print(torch.dequantize(vsnet_quantized.linear1.weight()))
print('')

Original weights: 
Parameter containing:
tensor([[-0.0176,  0.0058, -0.0045,  ..., -0.0188, -0.0143, -0.0386],
        [-0.0078, -0.0034, -0.0097,  ...,  0.0052, -0.0430, -0.0300],
        [ 0.0329, -0.0121, -0.0087,  ..., -0.0268,  0.0079, -0.0031],
        ...,
        [-0.0275, -0.0310, -0.0264,  ..., -0.0182,  0.0138,  0.0005],
        [ 0.0029, -0.0112,  0.0098,  ...,  0.0213,  0.0072,  0.0223],
        [ 0.0120,  0.0347, -0.0144,  ...,  0.0303, -0.0096,  0.0003]],
       requires_grad=True)

Dequantized weights: 
tensor([[-0.0173,  0.0058, -0.0058,  ..., -0.0202, -0.0144, -0.0376],
        [-0.0087, -0.0029, -0.0087,  ...,  0.0058, -0.0433, -0.0289],
        [ 0.0318, -0.0116, -0.0087,  ..., -0.0260,  0.0087, -0.0029],
        ...,
        [-0.0289, -0.0318, -0.0260,  ..., -0.0173,  0.0144,  0.0000],
        [ 0.0029, -0.0116,  0.0087,  ...,  0.0202,  0.0058,  0.0231],
        [ 0.0116,  0.0347, -0.0144,  ...,  0.0289, -0.0087,  0.0000]])



# Print size and accuracy of the quantized model

In [23]:
print('Size of the model after quantization')
print_size_of_model(vsnet_quantized)

print('Testing the model after quantization')
test(vsnet_quantized)

Size of the model after quantization
Size (KB): 95.556
Testing the model after quantization


Testing: 100%|███████████████████████████████| 313/313 [00:02<00:00, 119.96it/s]

Accuracy: 0.957
